<a href="https://colab.research.google.com/github/majiju/intro-cc-garcia_gaitan_ramirez/blob/main/L1_sesion2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L1 · Primer contacto — Sesión 2: **Responder**
### Introducción a las Ciencias de la Computación · UNAL Manizales

En la sesión 1 recibieron un mensaje. Hoy escriben uno: su especie le va a hablar a otra especie del salón, que lo va a leer **sin instrucciones**, mientras el canal le daña bits al azar.

Al final de la sesión viene **el día del contacto**: los mensajes se reparten y cada equipo intenta leer uno ajeno con las herramientas que construyó la semana pasada.

In [ ]:
EQUIPO = ["Camila Andrea García León", "María José Gaitán Hernández", "Mateo Ramírez Henao", ""]
MI_BASE = 8
NOMBRE_ESPECIE = "Klyntar"

## Momento 1 — Predecir *(en papel)*

**P4.** ¿Qué fracción de su mensaje creen que la otra especie va a entender? Un número entre 0 y 1, y qué sección creen que se va a perder primero.

**P5.** Si el canal daña 1 de cada 100 bits, ¿cuántos bits dañados esperan en su mensaje? ¿Y si lo mandan tres veces?

✍️
- P4:
- P5:

## 0. Sus herramientas

Peguen aquí **sus** funciones de la sesión 1 y de la tarea puente. Son suyas: se reutilizan, no se reescriben. Las pruebas de abajo verifican que llegaron completas.

In [3]:
# Peguen aquí: es_primo, rectangulos, filas_de, dibujar, leer_numero, columna,
#              a_base_rec (o a_base_ciclo) y de_base
MI_BASE = 8
NOMBRE_ESPECIE = "Klyntar"

def es_primo(n):
    es_primo_resultado = (n >= 2)
    d = 2
    while (d * d <= n) and es_primo_resultado:
        if n % d == 0:
            es_primo_resultado = False
        d = d + 1
    return es_primo_resultado

def rectangulos(n):
    parejas = []
    for i in range(1, int(n**0.5) + 1):
        if n % i == 0:
            parejas.append((i, n // i))
            if i != n // i:
                parejas.append((n // i, i))
    parejas.sort()
    return parejas

def filas_de(bits, columnas):
    renglones = []
    for i in range(0, len(bits), columnas):
        renglones.append(bits[i : i + columnas])
    return renglones

def dibujar(bits, columnas):
    for renglon in filas_de(bits, columnas):
        print(renglon.replace('1', '█').replace('0', ' '))

def leer_numero(lista_bits):
    acumulador = 0
    for bit in lista_bits:
        acumulador = (acumulador * 2) + bit
    return acumulador

def columna(bits, columnas, j, fila_desde, fila_hasta):
    resultado = []
    for fila in range(fila_desde, fila_hasta):
        indice = (fila * columnas) + j
        resultado.append(int(bits[indice]))
    return resultado

def a_base_ciclo(n, b):
    if n == 0:
        return [0]
    digitos = []
    while n > 0:
        digitos.append(n % b)
        n = n // b
    digitos.reverse()
    return digitos

def a_base_rec(n, b):
    if n < b:
        return [n]
    return a_base_rec(n // b, b) + [n % b]

def de_base(digitos, b):
    acumulador = 0
    for d in digitos:
        acumulador = acumulador * b + d
    return acumulador

In [4]:
# Pruebas de que las herramientas llegaron completas
assert es_primo(97) and not es_primo(1679)
assert rectangulos(12) == [(1, 12), (2, 6), (3, 4), (4, 3), (6, 2), (12, 1)]
assert filas_de("101100", 3) == ["101", "100"]
assert leer_numero([1, 0, 1, 1]) == 11
assert a_base_rec(1974, 6) == [1, 3, 0, 5, 0] and de_base([1, 3, 0, 5, 0], 6) == 1974
assert MI_BASE in (2, 3, 5, 6, 8, 12), "Escriban la base que les tocó."
print("✔ Herramientas listas. Especie de base", MI_BASE)

✔ Herramientas listas. Especie de base 8


## 1. ¿Cuántos bits por dígito?

Si cuentan en base 6, sus dígitos van de 0 a 5, y cada uno hay que escribirlo con bits. ¿Con cuántos? Es la desigualdad de la clase 3: con $k$ bits se escriben $2^k$ valores distintos, así que hace falta el menor $k$ con $2^k \geq b$.

- `bits_por_digito(b)` devuelve ese $k$.
- `digito_a_bits(d, ancho)` escribe un dígito con exactamente `ancho` bits, rellenando con ceros a la izquierda. Usen su `a_base_rec(d, 2)`.

In [5]:
def bits_por_digito(b):
    """CONTRATO. Recibe una base b >= 2. Devuelve el menor k con 2**k >= b."""
    # TU CÓDIGO AQUÍ
    k = 1
    while 2**k < b:
        k += 1
    return k


def digito_a_bits(d, ancho):
    """CONTRATO. Recibe un entero 0 <= d < 2**ancho. Devuelve una cadena de exactamente
    'ancho' caracteres '0'/'1' con d en binario, rellenada con ceros a la izquierda."""
    # TU CÓDIGO AQUÍ
    bits_lista = a_base_rec(d, 2)
    cadena = "".join(str(bit) for bit in bits_lista)
    return "0" * (ancho - len(cadena)) + cadena

In [6]:
# Pruebas
assert [bits_por_digito(b) for b in (2, 3, 5, 6, 8, 12)] == [1, 2, 3, 3, 3, 4]
assert digito_a_bits(5, 3) == "101" and digito_a_bits(1, 4) == "0001" and digito_a_bits(0, 2) == "00"
ANCHO = bits_por_digito(MI_BASE)
print("✔ Su especie escribe cada dígito con", ANCHO, "bits.")

✔ Su especie escribe cada dígito con 3 bits.


## 2. El mensaje, por secciones

**La representación del mensaje.** Un mensaje es una **lista de renglones**, y cada renglón es una cadena de `'0'` y `'1'` de exactamente `COLUMNAS` caracteres. Elijan `COLUMNAS` pensando en la pregunta de Arecibo: ¿cómo va a adivinar la otra especie el ancho del rectángulo?

Cada sección del mensaje es una **función** que devuelve una lista de renglones. Al final, `ensamblar` las pone una debajo de la otra. Si mañana quieren cambiar una sección, cambian una función y nada más: eso es pensar en módulos.

### La sección obligatoria: contar

`codificar_conteo(hasta)` escribe los números del 1 al `hasta` **en su base**, uno por renglón, cada dígito con `ANCHO` bits, alineados a la izquierda y rellenando el renglón con ceros. Ojo: deben contar al menos hasta un poco más allá de su base, para que se vea el momento en que aparece el "10".

In [8]:
COLUMNAS = 19    # elijan el ancho del mensaje
ANCHO = bits_por_digito(MI_BASE)


def renglon(contenido):
    """CONTRATO. Recibe una cadena de bits de longitud <= COLUMNAS.
    Devuelve esa cadena rellenada con ceros a la derecha hasta COLUMNAS caracteres."""
    assert len(contenido) <= COLUMNAS, f"El renglón no cabe: {len(contenido)} > {COLUMNAS}"
    return contenido + "0" * (COLUMNAS - len(contenido))


def codificar_conteo(hasta):
    """CONTRATO. Devuelve una lista de renglones: el renglón i tiene el número i
    (de 1 a 'hasta') escrito en MI_BASE, cada dígito con ANCHO bits."""
    # TU CÓDIGO AQUÍ (usen a_base_rec, digito_a_bits y renglon)
    lista_renglones = []
    for i in range(1, hasta + 1):
        digitos = a_base_rec(i, MI_BASE)
        contenido = "".join(digito_a_bits(d, ANCHO) for d in digitos)
        lista_renglones.append(renglon(contenido))
    return lista_renglones

In [9]:
# Pruebas
conteo = codificar_conteo(MI_BASE + 2)
assert len(conteo) == MI_BASE + 2 and all(len(r) == COLUMNAS for r in conteo)
assert conteo[0].startswith(digito_a_bits(1, ANCHO))
assert conteo[MI_BASE - 1].startswith(digito_a_bits(1, ANCHO) + digito_a_bits(0, ANCHO)), "El renglón de MI_BASE debe decir '10'."
print("✔ El conteo delata su base. Así se ve:")
dibujar("".join(conteo), COLUMNAS)

✔ El conteo delata su base. Así se ve:
  █                
 █                 
 ██                
█                  
█ █                
██                 
███                
  █                
  █  █             
  █ █              


### Sus secciones

Una función por sección, cada una con su contrato. Copien la plantilla tantas veces como secciones tengan.

In [12]:
def numero_en_mi_base(n):
    digitos = a_base_rec(n, MI_BASE)
    return "".join(digito_a_bits(d, ANCHO) for d in digitos)


def codificar_seccion_2():
    """CONTRATO. Qué dice: Identidad de los Klyntar (pictograma de núcleo simbionte
    con 8 nodos radiales y el número 8 escrito en base 8).
    Devuelve: lista de 5 renglones de COLUMNAS (19) bits."""
    ocho_octal = numero_en_mi_base(8)
    return [
        renglon("0000100010001000000"),
        renglon("0000010111010000000"),
        renglon("0000111111111" + ocho_octal),
        renglon("0000010111010000000"),
        renglon("0000100010001000000"),
    ]


def codificar_seccion_3():
    """CONTRATO. Qué dice: Dónde vivimos (Manizales a 2150 m de altitud).
    Dibuja una cordillera de tres montañas y el número 2150 en base 8 (4146_8).
    Devuelve: lista de 3 renglones de COLUMNAS (19) bits."""
    altitud_octal = numero_en_mi_base(2150)
    return [
        renglon("0001000" + altitud_octal),
        renglon("0101110000000000000"),
        renglon("1111111000000000000"),
    ]


def codificar_seccion_4():
    c_h = numero_en_mi_base(8) + numero_en_mi_base(10)
    n_o = numero_en_mi_base(4) + numero_en_mi_base(2)
    return [
        renglon("0110110" + c_h),
        renglon("0110110" + n_o),
    ]

In [14]:
def ensamblar(secciones):
    """CONTRATO. Recibe una lista de secciones (cada una, una lista de renglones).
    Devuelve una sola cadena de bits: todos los renglones, uno detrás de otro."""
    separador = renglon("")
    renglones_totales = []
    for i, sec in enumerate(secciones):
        renglones_totales.extend(sec)
        if i < len(secciones) - 1:
            renglones_totales.append(separador)
    return "".join(renglones_totales)


mensaje = ensamblar([
    codificar_conteo(MI_BASE + 2),
    codificar_seccion_2(),
    codificar_seccion_3(),
    codificar_seccion_4()
])
print("Longitud del mensaje:", len(mensaje), "bits")
print("Formas de dibujarlo:", rectangulos(len(mensaje)))
dibujar(mensaje, COLUMNAS)


Longitud del mensaje: 437 bits
Formas de dibujarlo: [(1, 437), (19, 23), (23, 19), (437, 1)]
  █                
 █                 
 ██                
█                  
█ █                
██                 
███                
  █                
  █  █             
  █ █              
                   
    █   █   █      
     █ ███ █       
    █████████  █   
     █ ███ █       
    █   █   █      
                   
   █   █    ██  ██ 
 █ ███             
███████            
                   
 ██ ██   █     █ █ 
 ██ ██ █   █       


### La forma

¿Su mensaje tiene una longitud que obligue a **una sola** manera de dibujarlo, como Arecibo? Si no, agreguen renglones de relleno hasta que el número de renglones lo permita. Pista: si `COLUMNAS` es primo y el número de renglones también, solo hay dos rectángulos posibles (y uno es el traspuesto del otro). Escriban `es_semiprimo(n)` con su `rectangulos` si les sirve.

In [15]:
def es_semiprimo(n):
    """CONTRATO. Recibe un entero n > 1. Devuelve True si n es el producto de dos
    números primos (distintos o iguales), lo que equivale a tener exactamente
    3 o 4 divisores positivos (es decir, 3 o 4 parejas en rectangulos(n))."""
    parejas = rectangulos(n)
    return len(parejas) in (3, 4)


# Verificamos si nuestro mensaje actual ya es semiprimo; si no lo fuera, agregamos relleno
renglones_actuales = filas_de(mensaje, COLUMNAS)
while not es_semiprimo(len(renglones_actuales) * COLUMNAS):
    renglones_actuales.append(renglon(""))

mensaje = "".join(renglones_actuales)

print("¿Es semiprimo?:", es_semiprimo(len(mensaje)))
print("Longitud final:", len(mensaje), "bits")
print("Formas posibles:", rectangulos(len(mensaje)))

¿Es semiprimo?: True
Longitud final: 437 bits
Formas posibles: [(1, 437), (19, 23), (23, 19), (437, 1)]


## 3. Las preguntas incómodas de la representación

Su mensaje tiene que responder **al menos dos** de estas. En cada una, escriban cómo lo resolvieron (o por qué no se puede).

1. **Ancho fijo.** Si cada número va en un renglón de `COLUMNAS` bits, ¿cuál es el número más grande que cabe en su base? ¿Qué pasa si quieren decir cuántos habitantes tiene Manizales? A esto se le llama **desbordamiento**.
2. **Decimales.** ¿Cómo dicen la altura de Manizales en metros si el otro no sabe qué es un metro, ni dónde va la coma? *(Pista real: en 1972, las placas de las sondas Pioneer usaron una propiedad del átomo de hidrógeno como regla para medir, porque el hidrógeno es igual en todo el universo.)*
3. **Texto.** ¿Por qué no pueden mandar la palabra "Manizales" en ASCII o en Unicode? ¿Qué **sí** pueden mandar?
4. **Forma.** ¿Cómo va a saber la otra especie cuántas columnas tiene su mensaje?

In [16]:
digitos_por_renglon = COLUMNAS // ANCHO
maximo = MI_BASE ** digitos_por_renglon - 1

poblacion_manizales = 458000
desborda = poblacion_manizales > maximo

altitud_metros = 2150
unidades_hidrogeno = round(altitud_metros / 0.21)

print(f"Dígitos en base {MI_BASE} que caben en {COLUMNAS} columnas:", digitos_por_renglon)
print("Número máximo sin desbordamiento:", maximo)
print(f"¿Cabe la población de Manizales ({poblacion_manizales}) en un renglón?:", not desborda)
print(f"Altitud de Manizales en unidades de hidrógeno (21 cm): {unidades_hidrogeno} (en base 8: {a_base_rec(unidades_hidrogeno, MI_BASE)})")

Dígitos en base 8 que caben en 19 columnas: 6
Número máximo sin desbordamiento: 262143
¿Cabe la población de Manizales (458000) en un renglón?: False
Altitud de Manizales en unidades de hidrógeno (21 cm): 10238 (en base 8: [2, 3, 7, 7, 6])


✍️
1.
2.
3.
4.

## 4. El canal

Entre las estrellas el mensaje se daña: la radiación voltea bits. Vamos a simular ese canal y a comparar **dos maneras** de proteger el mensaje.

- `ruido(bits, p)`: voltea cada bit con probabilidad `p` (usen `random.random() < p`).
- **Manera 1, paridad.** `con_paridad(bits, ancho)` agrega a cada renglón un bit extra para que el número de unos del renglón sea **par**. Del otro lado, `filas_danadas(bits, ancho)` dice qué renglones llegaron con número impar de unos. (Ojo: el ancho de cada renglón ahora es `ancho + 1`.)
- **Manera 2, repetición.** `repetir3(bits)` manda cada bit tres veces seguidas; `votar(bits)` toma cada grupo de tres y se queda con el que más aparece.

In [17]:
import random


def ruido(bits, p):
    """CONTRATO. Devuelve una copia de bits donde cada bit se volteó con probabilidad p."""
    resultado = []
    for b in bits:
        if random.random() < p:
            resultado.append("1" if b == "0" else "0")
        else:
            resultado.append(b)
    return "".join(resultado)


def con_paridad(bits, ancho):
    """CONTRATO. Parte bits en renglones de 'ancho' y a cada uno le agrega un bit de paridad
    al final, de modo que cada renglón (ahora de ancho + 1) tenga un número par de unos."""
    renglones = filas_de(bits, ancho)
    con_extra = []
    for r in renglones:
        bit_paridad = "1" if r.count("1") % 2 != 0 else "0"
        con_extra.append(r + bit_paridad)
    return "".join(con_extra)


def filas_danadas(bits, ancho):
    """CONTRATO. Recibe bits con paridad, en renglones de ancho + 1.
    Devuelve la lista de índices de los renglones con un número impar de unos."""
    renglones = filas_de(bits, ancho + 1)
    danadas = []
    for i, r in enumerate(renglones):
        if r.count("1") % 2 != 0:
            danadas.append(i)
    return danadas


def repetir3(bits):
    """CONTRATO. Devuelve bits con cada carácter repetido tres veces seguidas."""
    return "".join(b * 3 for b in bits)


def votar(bits):
    """CONTRATO. Recibe una cadena de longitud múltiplo de 3. Por cada grupo de tres,
    devuelve el bit que aparece al menos dos veces."""
    recuperados = []
    for i in range(0, len(bits), 3):
        grupo = bits[i : i + 3]
        recuperados.append("1" if grupo.count("1") >= 2 else "0")
    return "".join(recuperados)


In [18]:
# Pruebas
random.seed(0)
assert ruido("0101", 0) == "0101" and ruido("0101", 1) == "1010"
assert con_paridad("110100", 3) == "11001001"
assert filas_danadas("11001001", 3) == [] and filas_danadas("11011001", 3) == [0]
assert repetir3("10") == "111000" and votar("110001") == "10" and votar(repetir3("1011")) == "1011"
print("✔ El canal y las dos protecciones pasan las pruebas.")

✔ El canal y las dos protecciones pasan las pruebas.


### Medir, no suponer

Corran el canal **mil veces** con `p = 0.01` sobre su mensaje y cuenten:

- **Paridad:** ¿en cuántas transmisiones hubo al menos un error? ¿En cuántas `filas_danadas` lo detectó? ¿En cuántas hubo errores que **no** detectó? (¿Cómo puede pasar eso?)
- **Repetición:** ¿en cuántas transmisiones `votar` recuperó el mensaje **exacto**?
- ¿Cuántos bits cuesta cada manera, comparado con el mensaje original?

In [19]:
N = 1000
p = 0.01

con_error = detectadas = no_detectadas = 0
recuperadas_rep = 0

for _ in range(N):
    # Paridad
    protegido = con_paridad(mensaje, COLUMNAS)
    recibido = ruido(protegido, p)

    hubo_error = (recibido != protegido)
    hubo_deteccion = (len(filas_danadas(recibido, COLUMNAS)) > 0)

    if hubo_error:
        con_error += 1
        if hubo_deteccion:
            detectadas += 1
        else:
            no_detectadas += 1

    # Repetición
    protegido_rep = repetir3(mensaje)
    recibido_rep = ruido(protegido_rep, p)
    if votar(recibido_rep) == mensaje:
        recuperadas_rep += 1

print("Paridad   : con error", con_error, "| detectadas", detectadas, "| sin detectar", no_detectadas)
print("Repetición: recuperadas exactas", recuperadas_rep, "de", N)
print("Costo en bits: original", len(mensaje), "| paridad", len(con_paridad(mensaje, COLUMNAS)),
      "| repetición", len(repetir3(mensaje)))

Paridad   : con error 983 | detectadas 979 | sin detectar 4
Repetición: recuperadas exactas 887 de 1000
Costo en bits: original 437 | paridad 460 | repetición 1311


✍️ **¿Cuál de las dos protecciones elegirían para un mensaje que tarda 25 000 años en llegar, y por qué?** Con sus números. Piensen en la diferencia entre **detectar** un error y **corregirlo**.

## 5. Mandar el mensaje

Elijan su protección, pasen el mensaje por el canal **una sola vez** con `p = 0.01` y guárdenlo en `mensaje.txt`: una sola línea de ceros y unos, **sin** decir cuántas columnas tiene ni qué protección usaron. Entréguenle el archivo al monitor. Súbanlo también a su repositorio.

In [20]:
PROTECCION = "repeticion"

if PROTECCION == "paridad":
    enviado = ruido(con_paridad(mensaje, COLUMNAS), 0.01)
else:
    enviado = ruido(repetir3(mensaje), 0.01)

with open("mensaje.txt", "w") as f:
    f.write(enviado)
print("mensaje.txt listo:", len(enviado), "bits. Entréguenlo al monitor.")

mensaje.txt listo: 1311 bits. Entréguenlo al monitor.


## 6. El día del contacto

El monitor les entrega el `mensaje.txt` de **otra especie**. Tienen 30 minutos para leerlo **con sus herramientas de la sesión 1**: `len`, `rectangulos`, `dibujar`, `columna`, `leer_numero`, `de_base`. Si sirven para un mensaje que nunca habían visto, eran herramientas de verdad; si solo servían para Arecibo, eran un truco.

Súbanlo a Colab (ícono de carpeta → subir) con el nombre `recibido.txt`.

In [ ]:
with open("recibido.txt") as f:
    ajeno = f.read().strip()

print("1. Longitud cruda:", len(ajeno))
print("   Formas posibles (sin limpiar):", rectangulos(len(ajeno)))

# --- Opción A: ¿Viene protegido con repetición triple? (múltiplo de 3) ---
if len(ajeno) % 3 == 0:
    limpio_rep = votar(ajeno)
    print("\n[Posible REPETICIÓN TRIPLE detectada]")
    print("Longitud tras votar:", len(limpio_rep), "-> Formas:", rectangulos(len(limpio_rep)))

# --- Opción B: ¿Viene con bits de paridad? (cada fila tiene ancho + 1) ---
print("\n[Posibles dimensiones si trae PARIDAD (descontando 1 columna)]")
for filas, cols in rectangulos(len(ajeno)):
    if cols > 2:
        print(f"Si tuviera {cols} cols con paridad ({cols - 1} útiles x {filas} filas): total útil = {(cols - 1) * filas}")

✍️ **Lo que entendimos del mensaje de la especie ________ :**

- Ancho que dedujimos: ____ columnas, porque ____ .
- Base que dedujimos: ____ , porque ____ .
- Protección que creemos que usaron: ____ .
- Lo que dice cada sección:
  1.
  2.
  3.

## 7. La revelación

Cada especie lee en voz alta lo que entendió del mensaje que recibió, y la especie emisora revela su contrato de lectura. Anoten **de su propio mensaje**:

| Paso de nuestro contrato de lectura | ¿Lo dio la otra especie? | Si no, ¿por qué? |
|---|---|---|
| | | |
| | | |
| | | |

¿Hubo algo que la otra especie entendió **mal**, con toda seguridad? (Recuerden la palabra "arma" en *La llegada*.)

## Momento 3 — Explicar *(una página por equipo)*

1. **Confronten P4** con lo que de verdad se entendió de su mensaje. ¿Por qué la diferencia?
2. **La convención escondida:** ¿qué supusieron **sin darse cuenta** que la otra especie no tenía?
3. **Paridad contra repetición:** cuál eligieron y por qué, con las mediciones de la sección 4.
4. Una frase que empiece por *"Representar es..."*. Compárenla con la de la sesión 1: ¿cambió?

✍️

## La IA como objeto de estudio *(para la bitácora individual)*

Denle a su asistente el `mensaje.txt` de **otra especie** del salón (no el suyo), sin decirle nada más, y pídanle que lo traduzca. Ese mensaje no está en ningún lugar de internet: no lo puede reconocer, solo lo puede descifrar.

Comparen lo que dice el asistente con lo que reveló la especie emisora. ¿Tradujo con seguridad algo que no estaba ahí? Esa es la escena del "arma", y la acaban de medir. Y vuelvan a la pregunta de la sesión 1: ¿qué hizo con Arecibo, descifrar o reconocer?

## Foto de las predicciones

`foto_predicciones_sesion2.jpg`

---
### Antes de irse
1. **Ejecuten todo de arriba a abajo** (*Entorno de ejecución → Reiniciar y ejecutar todo*). Si algo falla, no está terminado.
2. **Guarden una copia en su repositorio `intro-cc`:** *Archivo → Guardar una copia en GitHub*. Cada integrante en el suyo.
3. La **bitácora de IA** es individual y va aparte (la plantilla es `bitacora_IA_L1.md`, en la carpeta `introduccion_cc` del repositorio de laboratorios).